**This notebook is for merging the previous test results from the pipelines and labeling the ground truth**

Uploading the files

In [43]:
import pandas as pd

tfidf = pd.read_csv("TD-IDF_TestingResults.csv")
bm25 = pd.read_csv("BM25_Results.csv")
sbert = pd.read_csv("SBERT_Results.csv")

Adding the model column and combining them

In [44]:
tfidf["model"] = "tfidf"
bm25["model"] = "bm25"
sbert["model"] = "sbert"

combined = pd.concat([tfidf, bm25, sbert], ignore_index=True)

Cleaning the preview column

In [45]:
combined["resume_preview"] = combined["resume_preview"].fillna("").astype(str).str.strip()
combined["job_text"] = combined["job_text"].fillna("").astype(str).str.strip()
combined["resume_domain"] = combined["resume_domain"].fillna("").astype(str).str.strip()

combined["has_preview"] = combined["resume_preview"] != ""

Sort rows with preview text and remove duplicates

In [46]:
combined = combined.sort_values(
    by=["job_id", "resume_index", "has_preview"],
    ascending=[True, True, False]
)

gt = combined.drop_duplicates(subset=["job_id", "resume_index"], keep="first")

Dropping duplicates

In [47]:
gt = gt[[
    "job_id",
    "job_text",
    "resume_index",
    "resume_domain",
    "resume_preview"
]].copy()

Adding the relevant column for labeling

In [48]:
gt["relevant"] = ""

Making the spreadsheet look nice and readable for labeling the ground truth

In [49]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

gt = gt.sort_values(by=["job_id", "resume_index"]).reset_index(drop=True)

wb = Workbook()
ws = wb.active
ws.title = "GroundTruth"

headers = ['job_id', 'job_text', 'resume_index', 'resume_domain', 'resume_preview', 'relevant']
col_widths = [14, 50, 14, 18, 60, 12]

header_fill = PatternFill("solid", start_color="2F5496")
header_font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
center = Alignment(horizontal="center", vertical="center", wrap_text=True)
left_wrap = Alignment(horizontal="left", vertical="top", wrap_text=True)
thin = Side(style="thin", color="CCCCCC")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for col, (h, w) in enumerate(zip(headers, col_widths), start=1):
    c = ws.cell(row=1, column=col, value=h)
    c.fill = header_fill
    c.font = header_font
    c.alignment = center
    c.border = border
    ws.column_dimensions[get_column_letter(col)].width = w
ws.row_dimensions[1].height = 22

alt = PatternFill("solid", start_color="F2F6FC")
for i, row in enumerate(gt.itertuples(index=False), start=2):
    fill = alt if i % 2 == 0 else PatternFill("solid", start_color="FFFFFF")
    ws.row_dimensions[i].height = 60
    for col, key in enumerate(headers, start=1):
        val = getattr(row, key) if key != 'relevant' else None
        c = ws.cell(row=i, column=col, value=val)
        c.font = Font(name="Arial", size=9)
        c.fill = fill
        c.border = border
        c.alignment = left_wrap if key in ('job_text', 'resume_preview') else Alignment(horizontal="center", vertical="center")

ws.freeze_panes = "A2"


And save

In [50]:
wb.save("GroundTruth.xlsx")
gt.head(20)

,job_id,job_text,resume_index,resume_domain,resume_preview,relevant
0,custom_0,"Looking for a data analyst with Python, SQL, d...",4,Banking,"M.S. Computer Engineering, University of Misso...",
1,custom_0,"Looking for a data analyst with Python, SQL, d...",19,Banking,"MBA, Human Resource Management – University of...",
2,custom_0,"Looking for a data analyst with Python, SQL, d...",20,Banking,,
3,custom_0,"Looking for a data analyst with Python, SQL, d...",40,Banking,,
4,custom_0,"Looking for a data analyst with Python, SQL, d...",48,Banking,"MBA, St. Mary’s College of California (2012). ...",
5,custom_0,"Looking for a data analyst with Python, SQL, d...",70,Finance,,
6,custom_0,"Looking for a data analyst with Python, SQL, d...",187,TEACHER,"Bachelor of Science in Reading, Delta State Un...",
7,custom_0,"Looking for a data analyst with Python, SQL, d...",235,Apparel,"B.S. Metallurgy & Materials Engineering, Color...",
8,custom_0,"Looking for a data analyst with Python, SQL, d...",242,Apparel,,
9,custom_0,"Looking for a data analyst with Python, SQL, d...",249,Apparel,"B.S. in Information Systems, Northeastern Univ...",
